In [2]:
import pandas as pd
import json
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
import json
from collections import Counter
import numpy as np
import pandas as pd
train_data = []
with open('../data/train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        train_data.append(json.loads(line))

test_data = []
with open('../data/test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))



all_data = train_data + test_data
print(f"Total: {len(all_data)}")

Total: 361579


In [4]:
# get all labels
all_labels = [article['label'] for article in all_data]

# check original distribution of combined data
label_counts = Counter(all_labels)
print("\nClass distribution in full dataset:")
for cls in range(6):
    count = label_counts[cls]
    pct = (count / len(all_labels)) * 100
    print(f"  Class {cls}: {count:6,} ({pct:5.1f}%)")


Class distribution in full dataset:
  Class 0: 87,600 ( 24.2%)
  Class 1: 107,687 ( 29.8%)
  Class 2: 101,369 ( 28.0%)
  Class 3: 47,910 ( 13.3%)
  Class 4: 15,135 (  4.2%)
  Class 5:  1,878 (  0.5%)


In [5]:
# split with stratification
indices = list(range(len(all_data)))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
)

print(f"\nNew split:")
print(f"  Train indices: {len(train_idx):,}")
print(f"   Test indices: {len(test_idx):,}")


New split:
  Train indices: 289,263
   Test indices: 72,316


In [6]:
# split the data
train_data_new = [all_data[i] for i in train_idx]
test_data_new = [all_data[i] for i in test_idx]

# get labels
train_labels_new = [article['label'] for article in train_data_new]
test_labels_new = [article['label'] for article in test_data_new]

print(f"\nNew datasets created:")
print(f"  Train: {len(train_data_new):,} articles")
print(f"  Dev: {len(test_data_new):,} articles")


New datasets created:
  Train: 289,263 articles
  Dev: 72,316 articles


In [7]:
# count new distributions
train_new_counts = Counter(train_labels_new)
test_new_counts = Counter(test_labels_new)

print("\nNew training distribution:")
for cls in range(6):
    count = train_new_counts[cls]
    pct = (count / len(train_labels_new)) * 100
    print(f"  Class {cls}: {count:6,} ({pct:5.1f}%)")

print("\nNew dev distribution:")
for cls in range(6):
    count = test_new_counts[cls]
    pct = (count / len(test_labels_new)) * 100
    print(f"  Class {cls}: {count:6,} ({pct:5.1f}%)")


New training distribution:
  Class 0: 70,080 ( 24.2%)
  Class 1: 86,150 ( 29.8%)
  Class 2: 81,095 ( 28.0%)
  Class 3: 38,328 ( 13.3%)
  Class 4: 12,108 (  4.2%)
  Class 5:  1,502 (  0.5%)

New dev distribution:
  Class 0: 17,520 ( 24.2%)
  Class 1: 21,537 ( 29.8%)
  Class 2: 20,274 ( 28.0%)
  Class 3:  9,582 ( 13.3%)
  Class 4:  3,027 (  4.2%)
  Class 5:    376 (  0.5%)


In [8]:
# save restratified data as dataframes
# convert to dataframe with text and label
train_df = pd.DataFrame({
    'text': [article['text'] for article in train_data_new],
    'label': [article['label'] for article in train_data_new]
})

test_df = pd.DataFrame({
    'text': [article['text'] for article in test_data_new],
    'label': [article['label'] for article in test_data_new]
})

# save to csv
train_df.to_csv('../output/train_restrat.csv', index=False)
test_df.to_csv('../output/test_restrat.csv', index=False)

print(f"Saved restratified data:")


Saved restratified data:


In [9]:
import pandas as pd

train_df = pd.read_csv('../output/train_restrat.csv')
test_df = pd.read_csv('../output/test_restrat.csv')

train_data_new = train_df.to_dict('records')
test_data_new = test_df.to_dict('records')

In [10]:
# feature extraction setup
import spacy
from collections import Counter
import os,time

# load spacy model
nlp = spacy.load('en_core_web_sm')
print("Model loaded")

Model loaded


In [11]:
# helper functions
def calculate_entropy(values):
    total = sum(values)
    if total == 0:
        return 0.0
    probs = [v/total for v in values if v > 0]
    return -sum(p * np.log2(p) for p in probs)

def tree_depth(token, max_depth=50):
    if not list(token.children) or max_depth <= 0:
        return 1
    return 1 + max((tree_depth(child, max_depth-1) for child in token.children), default=0)

In [12]:
def extract_features(doc):
    
    # get tokens
    tokens = [t for t in doc if not t.is_space]
    tokens_clean = [t for t in tokens if not t.is_punct]
    
    if len(tokens_clean) == 0:
        return [0.0] * 15
    
    total = len(tokens_clean)
    sentences = list(doc.sents)
    
    # F1: burstiness
    if len(sentences) >= 2:
        lengths = [len([t for t in s if not t.is_space]) for s in sentences]
        mean_len = np.mean(lengths)
        burstiness = np.std(lengths) / mean_len if mean_len > 0 else 0.0
    else:
        burstiness = 0.0
    
    # F2-6: POS ratios and diversity
    pos_counts = Counter(t.pos_ for t in tokens_clean)
    noun_ratio = (pos_counts.get('NOUN', 0) / total) * 100
    verb_ratio = (pos_counts.get('VERB', 0) / total) * 100
    adj_ratio = (pos_counts.get('ADJ', 0) / total) * 100
    adv_ratio = (pos_counts.get('ADV', 0) / total) * 100
    
    pos_vals = [pos_counts.get(p, 0) for p in ['NOUN','VERB','ADJ','ADV']]
    pos_diversity = calculate_entropy(pos_vals + [total - sum(pos_vals)])
    
    # F7: syntactic complexity
    depths = []
    for sent in sentences:
        root = [t for t in sent if t.dep_ == 'ROOT']
        if root:
            depths.append(tree_depth(root[0]))
    complexity = np.mean(depths) if depths else 0.0
    
    # F8: pronouns
    pronouns = sum(1 for t in tokens_clean if t.tag_ in ['PRP', 'PRP$'])
    pronoun_ratio = (pronouns / total) * 100
    
    # F9: contractions
    contractions = ["n't", "'re", "'ve", "'ll", "'d", "'m", "'s"]
    contraction_count = sum(1 for t in tokens if t.text.lower() in contractions)
    contraction_ratio = (contraction_count / len(tokens)) * 100
    
    # F10: hapax
    words = [t.text.lower() for t in tokens_clean]
    word_freq = Counter(words)
    hapax = sum(1 for count in word_freq.values() if count == 1)
    hapax_ratio = (hapax / len(words)) * 100 if words else 0.0
    
    # F11: formal transitions
    transitions = {'however', 'moreover', 'therefore', 'thus', 'consequently',
                   'nevertheless', 'furthermore', 'meanwhile', 'hence'}
    transition_count = sum(1 for t in tokens_clean if t.text.lower() in transitions)
    transition_ratio = (transition_count / total) * 100
    
    # F12: passive voice
    passive_sents = sum(1 for s in sentences if any(t.dep_ == 'auxpass' for t in s))
    passive_ratio = (passive_sents / len(sentences)) * 100 if sentences else 0.0
    
    # F13: repeated n-grams
    if len(words) >= 3:
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        repeated = sum(1 for c in trigram_counts.values() if c >= 2)
        ngram_ratio = (repeated / len(trigrams)) * 100
    else:
        ngram_ratio = 0.0
    
    # F14: style breaks
    style_breaks = 0
    if len(sentences) >= 4:
        seg_size = len(sentences) // 4
        segments = []
        for i in range(4):
            start = i * seg_size
            end = start + seg_size if i < 3 else len(sentences)
            seg = sentences[start:end]
            
            seg_lens = [len([t for t in s if not t.is_space]) for s in seg]
            seg_burst = np.std(seg_lens) / np.mean(seg_lens) if np.mean(seg_lens) > 0 else 0
            segments.append(seg_burst)
        
        for i in range(3):
            if abs(segments[i] - segments[i+1]) > 0.3:
                style_breaks += 1
    
    # F15: lexical density
    content_words = sum(1 for t in tokens_clean if t.pos_ in ['NOUN','VERB','ADJ','ADV'])
    lex_density = (content_words / total) * 100
    
    return [burstiness, noun_ratio, verb_ratio, adj_ratio, adv_ratio,
            pos_diversity, complexity, pronoun_ratio, contraction_ratio,
            hapax_ratio, transition_ratio, passive_ratio, ngram_ratio,
            style_breaks, lex_density]

In [13]:
# test the function on one article
sample_text = train_data_new[0]['text']
sample_doc = nlp(sample_text[:10000])  # limit length for speed
sample_features = extract_features(sample_doc)

print("Sample features extracted:")
feature_names = ['burstiness', 'noun_ratio', 'verb_ratio', 'adj_ratio', 'adv_ratio',
                 'pos_diversity', 'syntactic_complexity', 'pronouns', 'contractions',
                 'hapax', 'formal_transitions', 'passive', 'ngrams', 'style_breaks',
                 'lexical_density']
for name, value in zip(feature_names, sample_features):
    print(f"  {name:25s}: {value:.2f}")


Sample features extracted:
  burstiness               : 0.63
  noun_ratio               : 22.50
  verb_ratio               : 12.73
  adj_ratio                : 9.55
  adv_ratio                : 5.45
  pos_diversity            : 1.92
  syntactic_complexity     : 5.90
  pronouns                 : 8.64
  contractions             : 0.96
  hapax                    : 39.32
  formal_transitions       : 0.00
  passive                  : 9.52
  ngrams                   : 0.68
  style_breaks             : 0.00
  lexical_density          : 50.23


In [15]:
# extract features from restratified data
import time
import os

os.makedirs('../output', exist_ok=True)
start = time.time()

# check if we already started
train_checkpoint = '../output/train_checkpoint.csv'
test_checkpoint = '../output/test_checkpoint.csv'

# train set
print("Extracting train features...")
train_features = []
train_labels = []
start_idx = 0

if os.path.exists(train_checkpoint):
    temp = pd.read_csv(train_checkpoint)
    start_idx = len(temp)
    train_features = temp[feature_names].values.tolist()
    train_labels = temp['label'].values.tolist()
    print(f"Resuming from {start_idx:,}")

for i in range(start_idx, len(train_data_new), 100):
    end = min(i + 100, len(train_data_new))
    
    texts = [str(train_data_new[j]['text'] or '')[:100000] for j in range(i, end)]
    labels = [train_data_new[j]['label'] for j in range(i, end)]
    
    for doc in nlp.pipe(texts, batch_size=50):
        train_features.append(extract_features(doc))
    
    train_labels.extend(labels)
    
    # progress
    if len(train_features) % 1000 == 0:
        print(f"  {len(train_features):,} / {len(train_data_new):,}")
    
    # save checkpoint every 5000
    if len(train_features) % 5000 == 0:
        temp = pd.DataFrame(train_features, columns=feature_names)
        temp['label'] = train_labels
        temp.to_csv(train_checkpoint, index=False)
        print(f"  saved checkpoint")

print(f"Train done: {len(train_features):,} samples")

# dev set
print("\nExtracting test features...")
test_features = []
test_labels = []
start_idx = 0

if os.path.exists(test_checkpoint):
    temp = pd.read_csv(test_checkpoint)
    start_idx = len(temp)
    test_features = temp[feature_names].values.tolist()
    test_labels = temp['label'].values.tolist()
    print(f"Resuming from {start_idx:,}")

for i in range(start_idx, len(test_data_new), 100):
    end = min(i + 100, len(test_data_new))
    
    texts = [str(test_data_new[j]['text'] or '')[:100000] for j in range(i, end)]
    labels = [test_data_new[j]['label'] for j in range(i, end)]
    
    for doc in nlp.pipe(texts, batch_size=50):
        test_features.append(extract_features(doc))
    
    test_labels.extend(labels)
    
    if len(test_features) % 500 == 0:
        print(f"  {len(test_features):,} / {len(test_data_new):,}")
    
    # checkpoint every 2000
    if len(test_features) % 2000 == 0:
        temp = pd.DataFrame(test_features, columns=feature_names)
        temp['label'] = test_labels
        temp.to_csv(test_checkpoint, index=False)
        print(f"  saved checkpoint")

print(f"Test done: {len(test_features):,} samples")

# make final dataframes
train_df = pd.DataFrame(train_features, columns=feature_names)
train_df['label'] = train_labels

test_df = pd.DataFrame(test_features, columns=feature_names)
test_df['label'] = test_labels

# save final files
train_df.to_csv('../output/features_train.csv', index=False)
test_df.to_csv('../output/features_test.csv', index=False)

# cleanup checkpoints
if os.path.exists(train_checkpoint):
    os.remove(train_checkpoint)
if os.path.exists(test_checkpoint):
    os.remove(test_checkpoint)

elapsed = time.time() - start
print(f"\nTotal time: {int(elapsed/60)} minutes")
print(f"Saved features_train.csv ({len(train_df):,}) and features_test.csv ({len(test_df):,})")

Extracting train features...
Resuming from 240,000
  241,000 / 289,263
  242,000 / 289,263
  243,000 / 289,263
  244,000 / 289,263
  245,000 / 289,263
  saved checkpoint
  246,000 / 289,263
  247,000 / 289,263
  248,000 / 289,263
  249,000 / 289,263
  250,000 / 289,263
  saved checkpoint
  251,000 / 289,263
  252,000 / 289,263
  253,000 / 289,263
  254,000 / 289,263
  255,000 / 289,263
  saved checkpoint
  256,000 / 289,263
  257,000 / 289,263
  258,000 / 289,263
  259,000 / 289,263
  260,000 / 289,263
  saved checkpoint
  261,000 / 289,263
  262,000 / 289,263
  263,000 / 289,263
  264,000 / 289,263
  265,000 / 289,263
  saved checkpoint
  266,000 / 289,263
  267,000 / 289,263
  268,000 / 289,263
  269,000 / 289,263
  270,000 / 289,263
  saved checkpoint
  271,000 / 289,263
  272,000 / 289,263
  273,000 / 289,263
  274,000 / 289,263
  275,000 / 289,263
  saved checkpoint
  276,000 / 289,263
  277,000 / 289,263
  278,000 / 289,263
  279,000 / 289,263
  280,000 / 289,263
  saved checkpoi